# Python 문법 파일별 보강 3 · 27-1 여러 파일 읽기 · `glob`

**한 줄 요약** · txt 26개를 `glob` · `sorted`로 원본 순서대로 읽고, `os` · 읽는 단위 · 없는 파일 · 라벨 모으기까지 정리

- **원본** · `1.Python문법/01_실습문제/27-1.Read Multiple Files_Q.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/01_Python문법` 전체 정리 6-2 (여러 파일 읽기)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `glob("*.txt")`로 파일을 찾고 `sorted()`로 순서를 고정해 읽는다
2. `os.listdir` · `os.getcwd` · `glob`이 돌려주는 것의 차이를 말한다
3. `read()` · `readline()` · `readlines()` · 줄 반복의 결과를 구분한다
4. 없는 파일을 `exists()` · `FileNotFoundError`로 다룬다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">폴더 속 이미지 경로를 모을 때</td>
<td style="text-align: left;"><code>glob("*.jpg")</code> + <code>sorted()</code> (1-1)</td>
</tr>
<tr>
<td style="text-align: left;">라벨 · 캡션 파일을 한꺼번에 읽을 때</td>
<td style="text-align: left;"><code>Path.glob</code> · <code>read_text()</code> (2-2 · 3장)</td>
</tr>
<tr>
<td style="text-align: left;">텍스트 데이터를 줄 단위로 읽을 때</td>
<td style="text-align: left;"><code>for line in f</code> (2-1)</td>
</tr>
<tr>
<td style="text-align: left;">경로가 맞는지 확인할 때</td>
<td style="text-align: left;"><code>os.getcwd()</code> · <code>exists()</code> (1-2 · 2-2)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pf9_map.png" alt="27-1 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 실습 폴더 · 원본 문제</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">폴더 속 파일 모두 읽기</td>
<td style="text-align: left;">폴더 안 txt 파일을 빠짐없이, 원본 출력과 같은 순서로 읽으려면?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">읽는 방법과 없는 파일</td>
<td style="text-align: left;">파일 내용을 어떤 단위로 읽고, 없는 파일은 어떻게 다루는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">이미지마다 따로 있는 라벨 파일을 한 번에 읽어 <code>{이름: 라벨}</code>과 클래스 개수를 만들려면?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

원본 폴더 대신 1-1에서 **A.txt ~ Z.txt를 먼저 만들고** 읽음. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">없음 (기본 모듈 <code>glob</code> · <code>os</code> · <code>pathlib</code>)</td>
</tr>
<tr>
<td style="text-align: left;">seed · 장치</td>
<td style="text-align: left;">필요 없음</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">원본 출력의 단어 26개 → <code>실습파일/27-1/A.txt ~ Z.txt</code></td>
</tr>
</tbody>
</table>

In [1]:
# 실행 환경 확인 · 실습 폴더 준비 (원본의 ../source 폴더 대신 노트북 옆 실습파일/)
import sys                # 파이썬 버전 확인용
from pathlib import Path  # 경로를 객체로 다룸

BASE = Path("실습파일")                                         # 노트북 옆 실습 폴더
BASE.mkdir(exist_ok=True)                                       # 없으면 만들고, 있으면 통과
print("Python:", sys.version.split()[0])                        # 3.10 이상이면 됨
print("실습 폴더:", BASE.as_posix(), "/ 있음?", BASE.exists())  # 폴더 확인

Python: 3.14.6
실습 폴더: 실습파일 / 있음? True


### 원본 문제

- `/source/27-1/` 경로의 **모든 텍스트 파일**(txt)을 읽은 뒤 **리스트**로 출력하는 문제

```python
파일명 & 경로 = "../source/27-1/*.txt"

출력 = ['Python', 'JavaScript', 'PHP', 'Rust', 'elite', 'Solidity', 'Assembly', 'hamster', 'india',
        'january', 'kibana', 'lamada', 'monster', 'notion', 'orange', 'pokemon', 'query', 'range',
        'sonic', 'telegram', 'urban', 'village', 'world', 'x-ray', 'yellow', 'zigzag']
```

**원본 힌트**

- 다양한 확장자 형식의 파일 읽기는 중요 · `os` 패키지는 능숙해야 함
- `getcwd()` = 현재 작업 경로 · `listdir()` = 지정한 경로의 파일 · 폴더 전부
- `glob` 패키지도 사용 가능

---
## 1. 폴더 속 파일 모두 읽기

> **이 장의 흐름**
>
> - 원본 단어로 A.txt ~ Z.txt를 만들고 `glob`으로 읽어 원본 출력과 비교
> - 힌트의 `os.getcwd` · `os.listdir`과 비교

<img src="images/py1_glob.png" alt="glob으로 찾고 sorted로 순서 고정" width="763">

**파이프라인 위치** · ① 데이터 준비 · ② 불러오기 — 파일 목록 만들기

### 1-1. `A` 정답 · `glob`으로 찾아 순서대로 읽기 · `glob.glob("*.txt")` · `sorted()` <sub>원본 문제</sub>

> 🎯 **`*.txt`를 모두 찾아 원본 출력과 같은 리스트를 만들려면?**

💡 **쉽게 말하면**

- `glob("*.txt")`는 **이름 패턴으로 찾는 검색창** (`*` = 아무 글자).
- 찾은 순서는 보장되지 않아 `sorted()`로 줄 세운 뒤 하나씩 읽음.

**실행 흐름**

1. A.txt ~ Z.txt 만들기
2. *.txt 찾기 · 정렬
3. 하나씩 읽어 모으기
4. 원본 출력과 비교

**먼저 예측** · 읽은 리스트가 원본 출력과 같으려면 무엇이 꼭 필요할까? — 답은 코드 실행 결과 아래에

In [2]:
import glob               # 패턴으로 파일 찾기
from pathlib import Path  # 경로 객체

BASE = Path("실습파일")                                            # 노트북 옆 실습 폴더
folder = BASE / "27-1"                                             # 원본 경로 /source/27-1/
folder.mkdir(parents=True, exist_ok=True)                          # 폴더 만들기
words = ["Python", "JavaScript", "PHP", "Rust", "elite", "Solidity", "Assembly",  # A ~ G
         "hamster", "india", "january", "kibana", "lamada", "monster", "notion",  # H ~ N
         "orange", "pokemon", "query", "range", "sonic", "telegram", "urban",  # O ~ U
         "village", "world", "x-ray", "yellow", "zigzag"]          # V ~ Z (원본 출력 26개)
for letter, word in zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", words):      # 준비: 알파벳 ↔ 단어
    (folder / f"{letter}.txt").write_text(word, encoding="utf-8")  # 파일마다 단어 하나


# 정답: *.txt를 찾아 이름순으로 읽기
paths = sorted(glob.glob(str(folder / "*.txt")))       # 패턴 검색 → 정렬
result = []                                            # 읽은 단어를 모을 곳
for p in paths:                                        # 정렬된 순서대로
    with open(p, encoding="utf-8") as f:               # 읽기 모드
        result.append(f.read().strip())                # 앞뒤 공백 제거 후 추가
print("파일 개수:", len(result))                       # 26개
print("앞 5개:", result[:5], "/ 마지막:", result[-1])  # 일부만 출력
print("원본 출력과 같음:", result == words)            # 순서까지 같은지

파일 개수: 26
앞 5개: ['Python', 'JavaScript', 'PHP', 'Rust', 'elite'] / 마지막: zigzag
원본 출력과 같음: True


**결과** · 예측 정답은 `sorted()` — 이름순으로 고정해야 A ~ Z 순서

- `파일 개수: 26` — A.txt ~ Z.txt
- `앞 5개: ['Python', 'JavaScript', 'PHP', 'Rust', 'elite'] / 마지막: zigzag` — 파일 이름순 = 원본 출력 순서
- `원본 출력과 같음: True` — 개수 · 순서 모두 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>glob.glob("*.txt")</code></td>
<td style="text-align: left;">패턴에 맞는 <strong>경로</strong> 목록 (순서 보장 X)</td>
<td style="text-align: left;">특정 확장자만 찾을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>sorted(glob.glob(...))</code></td>
<td style="text-align: left;">이름순으로 <strong>순서 고정</strong></td>
<td style="text-align: left;"><strong>기본 선택</strong> (실행마다 같은 순서)</td>
</tr>
<tr>
<td style="text-align: left;"><code>glob("**/*.txt", recursive=True)</code></td>
<td style="text-align: left;">하위 폴더까지 검색</td>
<td style="text-align: left;">클래스별 폴더 아래 이미지</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 정렬 없이 쓰면 다른 PC · 다른 실행에서 순서가 바뀌어 라벨 짝이 어긋날 수 있음

> ✅ **한 줄 정리** · `glob("*.txt")`로 찾고 `sorted()`로 순서를 고정한 뒤 하나씩 읽으면 실행마다 같은 리스트가 나온다.

**딥러닝 코드에서는**

- `image_paths = sorted(glob.glob("data/train/*/*.jpg"))`

### 1-2. `B` 힌트의 `os` 함수 · `os.getcwd()` · `os.listdir()` · `endswith()` <sub>원본 힌트 · os 패키지</sub>

> 🎯 **`os.listdir`은 `glob`과 무엇이 다른가?**

💡 **쉽게 말하면**

- `getcwd()`는 **지금 서 있는 폴더**.
- `listdir()`은 폴더 안 **이름만 전부** (경로 · 순서 X) → 거르기 · 정렬은 직접.

**실행 흐름**

1. 현재 작업 경로 확인
2. listdir로 이름 목록
3. endswith로 txt만
4. 경로 붙이기

**먼저 예측** · `os.listdir(folder)`의 결과에는 폴더 경로가 붙어 있을까? — 답은 코드 실행 결과 아래에

In [3]:
import os  # 운영체제 기능 (경로 · 폴더)

print("작업 경로는 절대 경로?", os.path.isabs(os.getcwd()))              # getcwd(): 지금 작업 폴더
names = os.listdir(folder)  # 폴더 안 이름 목록 (경로 X · 순서 보장 X)
print("listdir 개수:", len(names), "/ 정렬 앞 3개:", sorted(names)[:3])  # 이름만 나옴
txt = [n for n in names if n.endswith(".txt")]                           # 확장자로 직접 거르기
print("txt만:", len(txt))                                                # 26개
print("경로 붙이기:", (folder / sorted(txt)[0]).as_posix())              # 이름 → 경로

작업 경로는 절대 경로? True
listdir 개수: 26 / 정렬 앞 3개: ['A.txt', 'B.txt', 'C.txt']
txt만: 26
경로 붙이기: 실습파일/27-1/A.txt


**결과** · 예측 정답은 아니요 — 이름만 (`A.txt`)

- `작업 경로는 절대 경로? True` — getcwd()는 C:/... 같은 전체 경로
- `listdir 개수: 26 / 정렬 앞 3개: ['A.txt', 'B.txt', 'C.txt']` — 경로 없이 이름만
- 거르기 · 경로 붙이기는 직접
  - `txt만: 26`
  - `경로 붙이기: 실습파일/27-1/A.txt`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>os.getcwd()</code></td>
<td style="text-align: left;">현재 작업 폴더 (절대 경로)</td>
<td style="text-align: left;">상대 경로가 어디 기준인지 확인</td>
</tr>
<tr>
<td style="text-align: left;"><code>os.listdir(폴더)</code></td>
<td style="text-align: left;"><strong>모든</strong> 이름 (경로 · 순서 X)</td>
<td style="text-align: left;">폴더 안 전체 목록</td>
</tr>
<tr>
<td style="text-align: left;"><code>glob.glob(패턴)</code></td>
<td style="text-align: left;">패턴으로 골라 <strong>경로</strong>까지</td>
<td style="text-align: left;">확장자 · 이름 패턴으로 고를 때 (기본 선택)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 노트북은 작업 폴더가 노트북 위치라 `"실습파일"` 같은 상대 경로는 노트북을 옮기면 달라짐

> ✅ **한 줄 정리** · `os.listdir`은 이름만 모두 주므로 거르기 · 정렬 · 경로 붙이기를 직접 하고, 보통은 `glob`이 더 간단하다.

**딥러닝 코드에서는**

- `classes = sorted(os.listdir("data/train"))` — 폴더 이름 = 클래스 목록

---
## 2. 읽는 방법과 없는 파일

> **이 장의 흐름**
>
> - 원본 힌트: 다양한 파일 읽기는 중요
> - 읽기 함수마다 결과 비교 · pathlib 한 줄 읽기 · 없는 파일 처리

<img src="images/pf9_read.png" alt="read · readline · readlines · for의 결과 차이" width="763">

**파이프라인 위치** · ② 불러오기 — 텍스트 · 라벨 파일

### 2-1. `B` 읽는 단위 비교 · `read()` · `readlines()` · `for line in f` <sub>원본 힌트 · 파일 읽기</sub>

> 🎯 **`read` · `readline` · `readlines` · 줄 반복은 각각 무엇을 돌려주는가?**

💡 **쉽게 말하면**

- `read()`는 **전체를 문자열 하나**로, `readline()`은 **한 줄만**.
- `readlines()`는 **줄 리스트**(줄바꿈 포함), `for line in f`는 **한 줄씩** 돌기.

**실행 흐름**

1. 3줄 파일 만들기
2. read()
3. readline()
4. readlines()
5. for + strip

**먼저 예측** · `readlines()`의 첫 원소는 `'cat'`일까 `'cat\n'`일까? — 답은 코드 실행 결과 아래에

In [4]:
multi = BASE / "multi.txt"                               # 여러 줄 파일
multi.write_text("cat\ndog\nbird\n", encoding="utf-8")   # 3줄 쓰기
with open(multi, encoding="utf-8") as f:                 # 매번 새로 열어 처음부터 읽기
    print("read():", repr(f.read()))                     # 전체를 문자열 하나로
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("readline():", repr(f.readline()))             # 첫 줄만
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("readlines():", f.readlines())                 # 줄 리스트 (줄바꿈 포함)
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("for + strip:", [line.strip() for line in f])  # 줄 단위 반복 (가장 흔함)

read(): 'cat\ndog\nbird\n'
readline(): 'cat\n'
readlines(): ['cat\n', 'dog\n', 'bird\n']
for + strip: ['cat', 'dog', 'bird']


**결과** · 예측 정답은 `'cat\n'` — 줄바꿈이 붙어 있음

- `read(): 'cat\ndog\nbird\n'` — 전체가 문자열 하나
- `readline(): 'cat\n'` — 한 줄만 · 줄바꿈 포함
- `readlines(): ['cat\n', 'dog\n', 'bird\n']` — 줄 리스트 · 줄바꿈 포함
- `for + strip: ['cat', 'dog', 'bird']` — `strip()`으로 줄바꿈 제거

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>read()</code></td>
<td style="text-align: left;">전체를 <strong>문자열 하나</strong>로</td>
<td style="text-align: left;">짧은 파일 · 단어 하나</td>
</tr>
<tr>
<td style="text-align: left;"><code>readlines()</code></td>
<td style="text-align: left;"><strong>줄 리스트</strong> (줄바꿈 포함)</td>
<td style="text-align: left;">줄 수가 적을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>for line in f</code></td>
<td style="text-align: left;">한 줄씩 읽음 (메모리 적게)</td>
<td style="text-align: left;">큰 파일 · 줄 단위 처리 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>read().splitlines()</code></td>
<td style="text-align: left;">줄 리스트 (줄바꿈 제거)</td>
<td style="text-align: left;">짧은 파일을 줄 목록으로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 같은 파일 객체로 `read()`를 두 번 부르면 두 번째는 `''` (이미 끝까지 읽음) → 다시 열어야 함

> ✅ **한 줄 정리** · 전체는 `read()`, 줄 목록은 `readlines()`, 큰 파일은 `for line in f`로 읽는다.

**딥러닝 코드에서는**

- 캡션 파일을 `for line in f:`로 한 줄씩 읽어 토큰화

### 2-2. `A` pathlib 한 줄 읽기와 없는 파일 · `Path.glob()` · `exists()` <sub>원본 힌트 · glob 패키지</sub>

> 🎯 **pathlib으로 더 짧게 읽고, 없는 파일은 어떻게 확인하는가?**

💡 **쉽게 말하면**

- `Path`는 **경로를 들고 다니는 객체** · `.glob()` · `.read_text()`를 바로 씀.
- 없는 파일은 **먼저 `exists()`로 확인**하거나 에러를 잡음.

**실행 흐름**

1. pathlib 한 줄 읽기
2. 1-1 결과와 비교
3. 없는 파일 exists()
4. FileNotFoundError 잡기

**먼저 예측** · 없는 파일을 `open()`으로 열면 어떤 에러가 날까? — 답은 코드 실행 결과 아래에

In [5]:
result2 = [p.read_text(encoding="utf-8").strip()   # 파일 내용 읽기 → 공백 제거
           for p in sorted(folder.glob("*.txt"))]  # Path.glob → 정렬
print("pathlib 결과 같음:", result2 == result)     # 1-1과 같은지


missing = folder / "없는파일.txt"                          # 존재하지 않는 경로
print("있음?", missing.exists())                           # 먼저 확인
try:
    open(missing, encoding="utf-8")                        # 없는 파일 열기
except FileNotFoundError as e:                             # 파일이 없음
    print(type(e).__name__, "/ 파일 이름:", missing.name)  # 에러 종류 · 파일 이름

pathlib 결과 같음: True
있음? False
FileNotFoundError / 파일 이름: 없는파일.txt


**결과** · 예측 정답은 `FileNotFoundError`

- `pathlib 결과 같음: True` — 두 줄로 1-1과 같은 결과
- `있음? False` — `exists()`로 미리 확인
- `FileNotFoundError / 파일 이름: 없는파일.txt` — 없는 파일을 열면 이 에러

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>glob.glob(str(경로))</code></td>
<td style="text-align: left;"><strong>문자열</strong> 경로 목록</td>
<td style="text-align: left;">옛 코드 · 문자열이 필요할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>Path(폴더).glob("*.txt")</code></td>
<td style="text-align: left;">경로 <strong>객체</strong> → <code>.read_text()</code> 바로 사용</td>
<td style="text-align: left;">경로를 다룰 일이 많을 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>exists()</code> 먼저</td>
<td style="text-align: left;">에러 전에 확인</td>
<td style="text-align: left;">있을 때만 처리</td>
</tr>
<tr>
<td style="text-align: left;"><code>except FileNotFoundError</code></td>
<td style="text-align: left;">열다가 실패하면 처리</td>
<td style="text-align: left;">없을 수도 있는 파일</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `except:`로 모든 에러를 잡으면 경로 오타 같은 버그도 숨음 → `FileNotFoundError`처럼 이름을 적기

> ✅ **한 줄 정리** · pathlib으로 짧게 읽고, 없는 파일은 `exists()`로 확인하거나 `FileNotFoundError`를 잡는다.

**딥러닝 코드에서는**

- `if not Path(ckpt).exists():` — 체크포인트가 없으면 처음부터 학습

---
## 3. 종합 실습 · 라벨 파일 모아 읽기

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - 이미지별 라벨 txt를 모으는 데이터 준비 단계를 가짜 파일로 흉내 냄

<img src="images/pf9_practice.png" alt="라벨 파일들을 읽어 이름 → 라벨 딕셔너리와 개수로" width="763">

**파이프라인 위치** · ① 데이터 준비 → ② 불러오기

> 🎯 **`img01.txt ~ img05.txt`(내용 = 라벨)를 읽어 딕셔너리와 클래스별 개수를 만들려면?**

💡 **쉽게 말하면**

- 파일 이름(**stem**)이 키, 파일 내용이 **라벨**.
- 다 모은 뒤 `Counter`로 클래스별 개수.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">가짜 라벨 파일 만들기</td>
<td style="text-align: left;">보강 2 파일 쓰기</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">정렬해서 찾기</td>
<td style="text-align: left;">1-1 <code>sorted</code> · 2-2 <code>Path.glob</code></td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">읽어 딕셔너리로</td>
<td style="text-align: left;">2-1 <code>strip()</code> · <code>p.stem</code></td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">클래스별 개수</td>
<td style="text-align: left;"><code>Counter</code></td>
</tr>
</tbody>
</table>

**먼저 예측** · `labels["img04"]`는 무엇일까? — 답은 코드 실행 결과 아래에

In [6]:
from collections import Counter  # 값 개수 세기


# 1단계: 이미지마다 라벨 파일 하나 (가짜 데이터, 끝에 줄바꿈 포함)
lab = BASE / "labels"                                                 # 라벨 폴더
lab.mkdir(parents=True, exist_ok=True)                                # 폴더 만들기
# 이름 → 라벨
fake = {"img01": "cat", "img02": "dog", "img03": "cat", "img04": "bird", "img05": "cat"}
for stem, label in fake.items():                                      # 파일마다
    (lab / f"{stem}.txt").write_text(label + "\n", encoding="utf-8")  # 라벨 한 줄 쓰기


# 2 · 3단계: 정렬해서 찾고 읽어 {이름: 라벨}
labels = {}                                                 # 결과 딕셔너리
for p in sorted(lab.glob("*.txt")):                         # 이름순
    labels[p.stem] = p.read_text(encoding="utf-8").strip()  # stem = 확장자 뺀 이름, 줄바꿈 제거


# 4단계: 결과 확인
print("읽은 개수:", len(labels), "/ 앞 3개:", list(labels)[:3])  # 5개 · 정렬 순서
print("img04:", labels["img04"])                                 # 이름으로 라벨 찾기
print("클래스별 개수:", dict(Counter(labels.values())))          # 라벨마다 몇 개

읽은 개수: 5 / 앞 3개: ['img01', 'img02', 'img03']
img04: bird
클래스별 개수: {'cat': 3, 'dog': 1, 'bird': 1}


**결과** · 예측 정답은 `bird`

- `읽은 개수: 5 / 앞 3개: ['img01', 'img02', 'img03']` — 정렬 순서대로 5개
- `img04: bird` — `strip()`으로 줄바꿈을 빼서 깔끔한 라벨
- `클래스별 개수: {'cat': 3, 'dog': 1, 'bird': 1}` — 클래스 불균형 확인

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>p.stem</code></td>
<td style="text-align: left;">확장자 뺀 이름 (<code>img01</code>)</td>
<td style="text-align: left;">이미지와 라벨 파일 짝짓기</td>
</tr>
<tr>
<td style="text-align: left;"><code>p.name</code></td>
<td style="text-align: left;">확장자 포함 (<code>img01.txt</code>)</td>
<td style="text-align: left;">파일 이름 그대로 쓸 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>.strip()</code></td>
<td style="text-align: left;">줄바꿈 · 공백 제거</td>
<td style="text-align: left;">읽은 라벨 정리 (항상)</td>
</tr>
<tr>
<td style="text-align: left;"><code>Counter(...)</code></td>
<td style="text-align: left;">값마다 개수</td>
<td style="text-align: left;">클래스 불균형 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `strip()`을 빼면 `'cat\n'`과 `'cat'`이 다른 클래스로 세어짐

> ✅ **한 줄 정리** · 라벨 파일은 `sorted(Path.glob())`로 모아 `stem`을 키, `strip()`한 내용을 값으로 담는다.

**딥러닝 코드에서는**

- YOLO 형식 라벨(이미지마다 txt)을 읽을 때 같은 패턴

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 정답</td>
<td style="text-align: left;">26개 · 원본 출력과 같음 (sorted 덕분)</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2. os</td>
<td style="text-align: left;">getcwd는 절대 경로 · listdir은 이름만</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3. 읽는 단위</td>
<td style="text-align: left;">read · readline · readlines · for의 차이</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4. pathlib · 없는 파일</td>
<td style="text-align: left;">한 줄 읽기 · exists() · FileNotFoundError</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 라벨 모으기</td>
<td style="text-align: left;">{이름: 라벨} 5개 · cat 3 · dog 1 · bird 1</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>sorted(glob(...))</code></td>
<td style="text-align: left;">찾고 순서 고정</td>
<td style="text-align: left;">이미지 경로 목록</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>Path.glob</code> · <code>read_text</code></td>
<td style="text-align: left;">경로 객체로 짧게 읽기</td>
<td style="text-align: left;">라벨 파일 읽기</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>strip()</code></td>
<td style="text-align: left;">줄바꿈 · 공백 제거</td>
<td style="text-align: left;">라벨 정리</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;"><code>os.listdir</code> · <code>getcwd</code></td>
<td style="text-align: left;">이름만 · 작업 폴더</td>
<td style="text-align: left;">클래스 폴더 목록</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>glob</code> 결과에 <code>sorted()</code>를 붙이는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>os.listdir</code>과 <code>glob</code>의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;"><code>readlines()</code>와 <code>read().splitlines()</code>의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">없는 파일을 열면?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">라벨 파일 이름에서 확장자를 뺀 이름은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `folder`에서 `A.txt` · `B.txt` · `C.txt` 세 파일만 읽어 리스트로 출력하기
2. `folder` 안 txt 파일 개수를 `Path.glob`으로 세기

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 이름이 `A` ~ `E`인 파일만 `glob`의 `[A-E]` 패턴으로 읽기
2. 1-1 결과에서 소문자로 시작하는 단어만 고르기
3. `multi.txt`의 줄 수를 `for` 반복으로 세기

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">순서가 보장되지 않아 실행마다 같은 순서로 고정</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">이름만 전부 vs 패턴으로 골라 경로까지</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">줄바꿈 포함 vs 제거</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">FileNotFoundError (미리 <code>exists()</code>로 확인 가능)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>p.stem</code></td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
from pathlib import Path

folder = Path("실습파일") / "27-1"
abc = [(folder / f"{c}.txt").read_text(encoding="utf-8") for c in "ABC"]
print(abc)  # ['Python', 'JavaScript', 'PHP']
print(len(list(folder.glob("*.txt"))))  # 26
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>sorted(folder.glob("[A-E].txt"))</code> → Python · JavaScript · PHP · Rust · elite</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>[w for w in result if w[0].islower()]</code> → <code>elite</code> ~ <code>zigzag</code> 20개</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>sum(1 for _ in open(multi, encoding="utf-8"))</code> → <code>3</code></td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>glob</code> 결과를 정렬 없이 써서 순서가 바뀜</td>
<td style="text-align: left;">glob이 이름순으로 준다고 착각</td>
<td style="text-align: left;">glob은 순서 보장 X</td>
<td style="text-align: left;"><code>sorted()</code>로 감쌌나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 폴더 안 파일 목록을 가져오는 방법은?**
> - `os.listdir`은 이름만 전부 돌려주고 `glob`은 패턴에 맞는 경로를 돌려줍니다.
> - 두 방법 모두 순서가 보장되지 않아 `sorted()`로 고정합니다.

**Q2. `readlines()`와 `for line in f`는 어떻게 다른가요?**
> - `readlines()`는 모든 줄을 리스트로 한 번에 메모리에 올립니다.
> - `for line in f`는 한 줄씩 읽어 큰 파일에서도 메모리를 적게 씁니다.

**Q3. `pathlib`을 쓰는 이유는?**
> - 경로를 문자열 대신 객체로 다뤄 `/`로 이어 붙이고 `.glob()` · `.read_text()` · `.exists()`를 바로 쓸 수 있습니다.
> - 운영체제마다 다른 경로 구분자도 알아서 처리합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>glob.glob()</code> · <code>sorted()</code></td>
<td style="text-align: left;">glob · Python 내장 함수</td>
<td style="text-align: left;">패턴으로 찾기 · 순서 고정</td>
<td style="text-align: left;">1-1 <code>sorted(glob.glob(str(folder / "*.txt")))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>open()</code> · <code>read()</code> · <code>strip()</code></td>
<td style="text-align: left;">Python 내장 함수 · 파일 · str</td>
<td style="text-align: left;">열기 · 전체 읽기 · 공백 제거</td>
<td style="text-align: left;">1-1 <code>f.read().strip()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>os.getcwd()</code> · <code>os.listdir()</code></td>
<td style="text-align: left;">os</td>
<td style="text-align: left;">작업 폴더 · 이름 목록</td>
<td style="text-align: left;">1-2 <code>os.listdir(folder)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>readline()</code> · <code>readlines()</code></td>
<td style="text-align: left;">파일 객체</td>
<td style="text-align: left;">한 줄 · 줄 리스트</td>
<td style="text-align: left;">2-1 <code>f.readlines()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>Path.glob()</code> · <code>read_text()</code> · <code>exists()</code> · <code>stem</code></td>
<td style="text-align: left;">pathlib</td>
<td style="text-align: left;">경로 객체로 찾기 · 읽기 · 확인</td>
<td style="text-align: left;">2-2 · 3 <code>p.stem</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>FileNotFoundError</code> · <code>Counter</code></td>
<td style="text-align: left;">예외 · collections</td>
<td style="text-align: left;">없는 파일 · 개수 세기</td>
<td style="text-align: left;">2-2 · 3 <code>Counter(labels.values())</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>glob</code> 결과는 <code>sorted()</code>로</td>
<td style="text-align: left;">정렬 없이 경로 순서 믿기</td>
</tr>
<tr>
<td style="text-align: left;">읽은 줄은 <code>strip()</code></td>
<td style="text-align: left;">같은 파일 객체로 <code>read()</code> 두 번</td>
</tr>
<tr>
<td style="text-align: left;">없을 수 있는 파일은 <code>exists()</code> 먼저</td>
<td style="text-align: left;"><code>except:</code>로 모든 에러 숨기기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>listdir</code> vs <code>glob</code></td>
<td style="text-align: left;">이름 전부 vs 패턴 · 경로</td>
<td style="text-align: left;">보통 glob</td>
</tr>
<tr>
<td style="text-align: left;"><code>readlines()</code> vs <code>for line in f</code></td>
<td style="text-align: left;">한 번에 vs 한 줄씩</td>
<td style="text-align: left;">큰 파일은 for</td>
</tr>
<tr>
<td style="text-align: left;"><code>p.stem</code> vs <code>p.name</code></td>
<td style="text-align: left;">확장자 없음 vs 포함</td>
<td style="text-align: left;">짝짓기는 stem</td>
</tr>
<tr>
<td style="text-align: left;"><code>read()</code> vs <code>read_text()</code></td>
<td style="text-align: left;">파일 객체 vs Path 객체</td>
<td style="text-align: left;">Path면 read_text</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">10분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 다시 풀기 · 2-1 결과를 실행 전에 예측</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">3장에 <code>img06.txt</code>를 빈 파일로 추가하고 어떻게 되는지 확인</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 보강 4(29-1) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">10분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 문제(`27-1.Read Multiple Files_Q.ipynb`)의 여러 파일 읽기와 힌트의 `os`
- 2장 · 힌트의 '다양한 파일 읽기' · `glob`을 읽는 단위 · pathlib으로 확장
- 원본의 `../source/27-1/` 대신 1-1에서 `실습파일/27-1/A.txt ~ Z.txt`를 만들어 읽음
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">문제 · 모든 txt 읽기 · 출력 리스트</td>
<td style="text-align: left;">정답 · glob으로 찾아 순서대로 읽기</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">힌트 · getcwd · listdir</td>
<td style="text-align: left;">os · getcwd · listdir</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">힌트 · 다양한 파일 읽기</td>
<td style="text-align: left;">읽는 단위 비교</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">힌트 · glob 패키지</td>
<td style="text-align: left;">pathlib 한 줄 읽기와 없는 파일</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">문제 · /source/27-1/ 경로</td>
<td style="text-align: left;">1-1에서 실습 파일 만들기</td>
<td style="text-align: left;">1-1</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [9]:
import glob               # 패턴으로 파일 찾기
from pathlib import Path  # 경로 객체
import os                 # 운영체제 기능 (경로 · 폴더)


# ---------- 1-1 ----------

BASE = Path("실습파일")                                            # 노트북 옆 실습 폴더
folder = BASE / "27-1"                                             # 원본 경로 /source/27-1/
folder.mkdir(parents=True, exist_ok=True)                          # 폴더 만들기
words = ["Python", "JavaScript", "PHP", "Rust", "elite", "Solidity", "Assembly",  # A ~ G
         "hamster", "india", "january", "kibana", "lamada", "monster", "notion",  # H ~ N
         "orange", "pokemon", "query", "range", "sonic", "telegram", "urban",  # O ~ U
         "village", "world", "x-ray", "yellow", "zigzag"]          # V ~ Z (원본 출력 26개)
for letter, word in zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", words):      # 준비: 알파벳 ↔ 단어
    (folder / f"{letter}.txt").write_text(word, encoding="utf-8")  # 파일마다 단어 하나


# 정답: *.txt를 찾아 이름순으로 읽기
paths = sorted(glob.glob(str(folder / "*.txt")))       # 패턴 검색 → 정렬
result = []                                            # 읽은 단어를 모을 곳
for p in paths:                                        # 정렬된 순서대로
    with open(p, encoding="utf-8") as f:               # 읽기 모드
        result.append(f.read().strip())                # 앞뒤 공백 제거 후 추가
print("파일 개수:", len(result))                       # 26개
print("앞 5개:", result[:5], "/ 마지막:", result[-1])  # 일부만 출력
print("원본 출력과 같음:", result == words)            # 순서까지 같은지


# ---------- 1-2 ----------

print("작업 경로는 절대 경로?", os.path.isabs(os.getcwd()))              # getcwd(): 지금 작업 폴더
names = os.listdir(folder)  # 폴더 안 이름 목록 (경로 X · 순서 보장 X)
print("listdir 개수:", len(names), "/ 정렬 앞 3개:", sorted(names)[:3])  # 이름만 나옴
txt = [n for n in names if n.endswith(".txt")]                           # 확장자로 직접 거르기
print("txt만:", len(txt))                                                # 26개
print("경로 붙이기:", (folder / sorted(txt)[0]).as_posix())              # 이름 → 경로

파일 개수: 26
앞 5개: ['Python', 'JavaScript', 'PHP', 'Rust', 'elite'] / 마지막: zigzag
원본 출력과 같음: True
작업 경로는 절대 경로? True
listdir 개수: 26 / 정렬 앞 3개: ['A.txt', 'B.txt', 'C.txt']
txt만: 26
경로 붙이기: 실습파일/27-1/A.txt


In [10]:
# ---------- 2-1 ----------
multi = BASE / "multi.txt"                               # 여러 줄 파일
multi.write_text("cat\ndog\nbird\n", encoding="utf-8")   # 3줄 쓰기
with open(multi, encoding="utf-8") as f:                 # 매번 새로 열어 처음부터 읽기
    print("read():", repr(f.read()))                     # 전체를 문자열 하나로
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("readline():", repr(f.readline()))             # 첫 줄만
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("readlines():", f.readlines())                 # 줄 리스트 (줄바꿈 포함)
with open(multi, encoding="utf-8") as f:                 # 다시 열기
    print("for + strip:", [line.strip() for line in f])  # 줄 단위 반복 (가장 흔함)


# ---------- 2-2 ----------
result2 = [p.read_text(encoding="utf-8").strip()   # 파일 내용 읽기 → 공백 제거
           for p in sorted(folder.glob("*.txt"))]  # Path.glob → 정렬
print("pathlib 결과 같음:", result2 == result)     # 1-1과 같은지


missing = folder / "없는파일.txt"                          # 존재하지 않는 경로
print("있음?", missing.exists())                           # 먼저 확인
try:
    open(missing, encoding="utf-8")                        # 없는 파일 열기
except FileNotFoundError as e:                             # 파일이 없음
    print(type(e).__name__, "/ 파일 이름:", missing.name)  # 에러 종류 · 파일 이름

read(): 'cat\ndog\nbird\n'
readline(): 'cat\n'
readlines(): ['cat\n', 'dog\n', 'bird\n']
for + strip: ['cat', 'dog', 'bird']
pathlib 결과 같음: True
있음? False
FileNotFoundError / 파일 이름: 없는파일.txt


In [11]:
# ---------- 3. 종합 실습 ----------
from collections import Counter  # 값 개수 세기


# 1단계: 이미지마다 라벨 파일 하나 (가짜 데이터, 끝에 줄바꿈 포함)
lab = BASE / "labels"                                                 # 라벨 폴더
lab.mkdir(parents=True, exist_ok=True)                                # 폴더 만들기
# 이름 → 라벨
fake = {"img01": "cat", "img02": "dog", "img03": "cat", "img04": "bird", "img05": "cat"}
for stem, label in fake.items():                                      # 파일마다
    (lab / f"{stem}.txt").write_text(label + "\n", encoding="utf-8")  # 라벨 한 줄 쓰기


# 2 · 3단계: 정렬해서 찾고 읽어 {이름: 라벨}
labels = {}                                                 # 결과 딕셔너리
for p in sorted(lab.glob("*.txt")):                         # 이름순
    labels[p.stem] = p.read_text(encoding="utf-8").strip()  # stem = 확장자 뺀 이름, 줄바꿈 제거


# 4단계: 결과 확인
print("읽은 개수:", len(labels), "/ 앞 3개:", list(labels)[:3])  # 5개 · 정렬 순서
print("img04:", labels["img04"])                                 # 이름으로 라벨 찾기
print("클래스별 개수:", dict(Counter(labels.values())))          # 라벨마다 몇 개

읽은 개수: 5 / 앞 3개: ['img01', 'img02', 'img03']
img04: bird
클래스별 개수: {'cat': 3, 'dog': 1, 'bird': 1}
